# PHASE 3B - PARALLEL PART 2/4: SEEDS 67 TO 91
Evaluates 25 placebo vectors (Seeds 67 to 91) with prompt_len steering hook in **~1.5 minutes** on Kaggle GPU.

In [1]:
import os, json, torch, torch.nn.functional as F, pandas as pd, sys
from transformers import AutoTokenizer, AutoModelForCausalLM

print('PyTorch Version:', torch.__version__, flush=True)
print('CUDA Available:', torch.cuda.is_available(), flush=True)
if torch.cuda.is_available():
    print('Device Count:', torch.cuda.device_count(), flush=True)
    for d in range(torch.cuda.device_count()):
        print(f'  GPU {d}:', torch.cuda.get_device_name(d), flush=True)

PyTorch Version: 2.10.0+cu128
CUDA Available: True
Device Count: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [2]:
# Load Qwen2.5-7B-Instruct Model across Multi-GPU
model_id = 'Qwen/Qwen2.5-7B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto' if torch.cuda.is_available() else None,
    trust_remote_code=True
)
model.eval()
print('✅ Model successfully loaded across multi-GPU!', flush=True)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Model successfully loaded across multi-GPU!


In [3]:
# Generate Placebo Vectors for Seeds 67 to 91
hidden_dim = model.config.hidden_size
placebo_vectors = []
for seed in range(67, 91 + 1):
    torch.manual_seed(seed)
    v_rand = torch.randn(hidden_dim, dtype=torch.float32)
    v_rand = v_rand / v_rand.norm(p=2)
    placebo_vectors.append({'seed': seed, 'vector': v_rand})
print(f'✅ Generated {len(placebo_vectors)} isotropic random unit vectors (Seeds 67 to 91).', flush=True)

✅ Generated 25 isotropic random unit vectors (Seeds 67 to 91).


In [4]:
# Load 500 Test Questions & Define Correct Prompt-Token Position Hook Scoring
possible_paths = [
    '/kaggle/input/datasets/thanhtranguyn/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/datasets/thanhtranguyn/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    'data/vietnamese_medical_halueval_15k_specialized.json'
]
dataset_path = None
for p in possible_paths:
    if os.path.exists(p):
        dataset_path = p
        break

with open(dataset_path, 'r', encoding='utf-8') as f:
    full_ds = json.load(f)

test_500 = full_ds[-500:]

def compute_seq_logprob_with_steering(model, tokenizer, prompt_str, target_str, v_steer, alpha=1.0, layer_idx=16):
    full_text = prompt_str + '\n' + target_str
    enc_prompt = tokenizer(prompt_str, return_tensors='pt')
    enc_full = tokenizer(full_text, return_tensors='pt').to(model.device)
    prompt_len = enc_prompt.input_ids.shape[1]
    labels = enc_full.input_ids.clone()
    labels[:, :prompt_len] = -100
    def hook_fn(module, input, output):
        if isinstance(output, tuple):
            h = output[0]
            v_dev = v_steer.to(device=h.device, dtype=h.dtype)
            h[:, prompt_len - 1, :] += alpha * v_dev
            return (h,) + output[1:]
        else:
            v_dev = v_steer.to(device=output.device, dtype=output.dtype)
            output[:, prompt_len - 1, :] += alpha * v_dev
            return output
    handle = model.model.layers[layer_idx].register_forward_hook(hook_fn)
    with torch.no_grad():
        outputs = model(**enc_full)
        logits = outputs.logits
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), reduction='sum')
    handle.remove()
    return -loss.item()

print(f'✅ Loaded {len(test_500)} test items with prompt_len steering hook!', flush=True)

✅ Loaded 500 test items with prompt_len steering hook!


In [5]:
# Run Parallel Part Evaluation
csv_filename = 'placebo_part2.csv'
completed_seeds = set()

if os.path.exists(csv_filename):
    df_existing = pd.read_csv(csv_filename)
    completed_seeds = set(df_existing['seed'].tolist())

if not os.path.exists(csv_filename):
    df_init = pd.DataFrame(columns=['seed', 'accuracy'])
    df_init.to_csv(csv_filename, index=False)

print('========================================================================', flush=True)
print('🚀 STARTING PART 2/4 BENCHMARK (SEEDS 67 TO 91):', flush=True)
print('========================================================================', flush=True)

for idx, vec_item in enumerate(placebo_vectors, 1):
    seed = vec_item['seed']
    if seed in completed_seeds:
        print(f'  [Part 2 | {idx:02d}/25] Seed {seed:03d} -> ALREADY COMPLETED.', flush=True)
        continue
    v_raw = vec_item['vector']
    correct_count = 0
    for item in test_500:
        q_str = item.get('question', item.get('prompt', ''))
        gold_str = item.get('right_answer', item.get('gold_answer', item.get('gold_ref', '')))
        hal_str = item.get('hallucinated_answer', item.get('hal_ref', ''))
        lp_gold = compute_seq_logprob_with_steering(model, tokenizer, q_str, gold_str, v_raw)
        lp_hal = compute_seq_logprob_with_steering(model, tokenizer, q_str, hal_str, v_raw)
        if lp_gold > lp_hal:
            correct_count += 1
    acc = (correct_count / len(test_500)) * 100.0
    df_new = pd.DataFrame([{'seed': seed, 'accuracy': acc}])
    df_new.to_csv(csv_filename, mode='a', header=False, index=False)
    completed_seeds.add(seed)
    print(f'  [Part 2 | {idx:02d}/25] Seed {seed:03d} | Acc: {acc:.2f}% ({correct_count}/500) -> SAVED!', flush=True)

df_res = pd.read_csv(csv_filename)
print('========================================================================', flush=True)
print('📊 PART 2/4 BENCHMARK COMPLETE (placebo_part2.csv)!', flush=True)
print(f'   Mean Accuracy: {df_res["accuracy"].mean():.2f}% ± {df_res["accuracy"].std():.2f}%', flush=True)
print('========================================================================', flush=True)

🚀 STARTING PART 2/4 BENCHMARK (SEEDS 67 TO 91):
  [Part 2 | 01/25] Seed 067 | Acc: 33.00% (165/500) -> SAVED!
  [Part 2 | 02/25] Seed 068 | Acc: 32.80% (164/500) -> SAVED!
  [Part 2 | 03/25] Seed 069 | Acc: 33.00% (165/500) -> SAVED!
  [Part 2 | 04/25] Seed 070 | Acc: 32.60% (163/500) -> SAVED!
  [Part 2 | 05/25] Seed 071 | Acc: 33.00% (165/500) -> SAVED!
  [Part 2 | 06/25] Seed 072 | Acc: 32.60% (163/500) -> SAVED!
  [Part 2 | 07/25] Seed 073 | Acc: 33.00% (165/500) -> SAVED!
  [Part 2 | 08/25] Seed 074 | Acc: 33.20% (166/500) -> SAVED!
  [Part 2 | 09/25] Seed 075 | Acc: 33.00% (165/500) -> SAVED!
  [Part 2 | 10/25] Seed 076 | Acc: 33.00% (165/500) -> SAVED!
  [Part 2 | 11/25] Seed 077 | Acc: 32.60% (163/500) -> SAVED!
  [Part 2 | 12/25] Seed 078 | Acc: 32.40% (162/500) -> SAVED!
  [Part 2 | 13/25] Seed 079 | Acc: 32.60% (163/500) -> SAVED!
  [Part 2 | 14/25] Seed 080 | Acc: 32.80% (164/500) -> SAVED!
  [Part 2 | 15/25] Seed 081 | Acc: 33.20% (166/500) -> SAVED!
  [Part 2 | 16/25] See